# Porous Materials BET Surface Area Extraction Pipeline

End-to-end pipeline for extracting synthesis procedures **and** BET surface area / adsorption data
from porous materials papers (MOFs, zeolites, aerogels, porous carbons, etc.).

## Pipeline Overview

```
PDF (bytes)
    ↓
[0. PDF Extraction]             → Markdown text + embedded figures (base64)
    ↓
[1. Material Extraction]        → List of synthesized porous materials
    ↓
[2. Synthesis Extraction]       → GeneralSynthesisOntology per material
    ↓
[3. Adsorption Text Extraction] → BET surface area + gas adsorption coverage per material  ★
    ↓
[4. Figure Extraction]          → Segmented subfigures (Florence-2)
    ↓
[5. Plot Data Extraction]       → Coordinate data from plots via Claude VLM
    ↓
[6. Plot Filtering]             → Keep only adsorption/isotherm plots
    ↓
[7. VLM Plot Extraction]        → Adsorption values from isotherm plots via Claude
    ↓
[8. Link Series → Materials]
    ↓
[9. Aggregate & Save]
```

## What this notebook extracts (per material)
- **Synthesis procedure** (GeneralSynthesisOntology: precursors, conditions, method)
- **BET surface area** (m²/g) reported in text
- **Gas adsorption coverage** (maximum loading per gram) reported in text
- **Isotherm plot data** (pressure vs. uptake coordinates, linked to each material)
- All results saved per-material to JSON + aggregated to a master CSV


## Setup

In [ ]:
# ==============================================================================
# USER CONFIGURATION
# ==============================================================================

# Path to your porosity paper (PDF or markdown)
# INPUT_PATH = "/Users/valeriegentzke/Desktop/try_paper.pdf"  # <-- CHANGE THIS
INPUT_PATH = "test_paper.pdf"  # or .md file with embedded images

# Output directory for results (inside the PDF papers folder)
PDF_DIR = r"./"
OUTPUT_DIR = f"{PDF_DIR}"

# Master CSV for multi-paper aggregation (appended after each run)
MASTER_CSV = f"{OUTPUT_DIR}/por_master.csv"

# Models
GEMINI_MODEL = "gemini-2.5-flash-lite"
# "gemini-3.0-flash"
# For synthesis / porosity text extraction
CLAUDE_MODEL = (
    "claude-sonnet-4-20250514"  # For VLM plot data + Porosity extraction
)
LINKER_MODEL = "gemini-3.0-flash"  # For series-to-material matching

# Set to True to skip figure extraction (synthesis + porosity text only)
SKIP_FIGURES = False

In [ ]:
import sys

print(sys.executable)

In [ ]:
# Load environment and imports
import json
import logging
import os
import re
import ssl
import sys
import warnings
from pathlib import Path

# !{sys.executable} -m pip install transformers anthropic dspy
from dotenv import load_dotenv


# ── Fast dependency check (fail early, not at Step 4) ──
def _check_dependencies():
    """Verify critical dependencies are installed and compatible before running the pipeline."""
    errors = []

    # Check transformers + CLIP (needed for Florence-2 figure extraction)
    try:
        from transformers import CLIPImageProcessor
    except (ImportError, ModuleNotFoundError):
        try:
            import transformers

            ver = transformers.__version__
        except Exception:
            ver = "unknown"
        errors.append(
            f"transformers.CLIPImageProcessor not found (transformers=={ver}).\n"
            f"   Fix: pip install --upgrade transformers\n"
            f"   Or:  uv pip install --upgrade transformers"
        )

    # Check anthropic SDK (needed for Claude VLM calls)
    try:
        import anthropic
    except ImportError:
        errors.append("anthropic SDK not installed. Fix: pip install anthropic")

    # Check dspy (needed for text extraction)
    try:
        import dspy
    except ImportError:
        errors.append("dspy not installed. Fix: pip install dspy")

    if errors:
        print("=" * 60)
        print("DEPENDENCY CHECK FAILED")
        print("=" * 60)
        for e in errors:
            print(f"  ✗ {e}")
        print("=" * 60)
        raise ImportError(
            "Fix the above dependencies before running the pipeline."
        )
    else:
        print(
            "[OK] Dependency check passed (transformers/CLIP, anthropic, dspy)"
        )


_check_dependencies()

# Fix SSL certificate issue for uv-managed Python on macOS
# The httpx client (used by anthropic SDK) may fail to find SSL certs
_ssl_cert = ssl.get_default_verify_paths().cafile
if _ssl_cert and os.path.exists(_ssl_cert):
    os.environ.setdefault("SSL_CERT_FILE", _ssl_cert)
    os.environ.setdefault("SSL_CERT_DIR", os.path.dirname(_ssl_cert))

src_path = Path("../../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

env_path = Path("../../.env")
load_dotenv(env_path, override=True)

warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

logging.getLogger("pydantic").setLevel(logging.ERROR)
logging.getLogger("LiteLLM").setLevel(logging.ERROR)
logging.getLogger("litellm").setLevel(logging.ERROR)

print("[OK] Environment loaded")
print(f"[OK] src path: {src_path}")
print(f"[OK] SSL_CERT_FILE: {os.environ.get('SSL_CERT_FILE', 'not set')}")

In [ ]:
import os

print(os.getcwd())

---
## Step 0: Load Paper Text

In [ ]:
from llm_synthesis.models.paper import Paper

# SI file detection helpers (same as synthesis_with_performance.ipynb)
SI_PATTERNS = [
    "_SI",
    "-SI",
    "_si",
    "-si",
    "_Supporting",
    "_supporting",
    "_Supplementary",
    "_supplementary",
    "_supp",
    "_Supp",
]


def find_si_file(main_paper_path: Path) -> Path | None:
    parent_dir = main_paper_path.parent
    main_stem = main_paper_path.stem
    for pattern in SI_PATTERNS:
        for ext in [".pdf", ".md", ".txt"]:
            si_path = parent_dir / f"{main_stem}{pattern}{ext}"
            if si_path.exists():
                return si_path
    return None


def load_file_text(path: Path, pdf_extractor=None) -> str:
    suffix = path.suffix.lower()
    if suffix == ".pdf":
        if pdf_extractor is None:
            from llm_synthesis.transformers.pdf_extraction import (
                MistralPDFExtractor,
            )

            pdf_extractor = MistralPDFExtractor(structured=False)
        with open(path, "rb") as f:
            return pdf_extractor.forward(f.read())
    elif suffix in [".md", ".txt"]:
        with open(path, errors="replace") as f:
            return f.read()
    else:
        raise ValueError(f"Unsupported file type: {suffix}")


# Load main paper
input_path = Path(INPUT_PATH)
pdf_extractor = None

if input_path.suffix.lower() == ".pdf":
    print(f"Extracting text from PDF: {input_path.name}")
    from llm_synthesis.transformers.pdf_extraction import MistralPDFExtractor

    pdf_extractor = MistralPDFExtractor(structured=False)
    paper_text = load_file_text(input_path, pdf_extractor)
    print(f"   Main paper: {len(paper_text):,} characters")
elif input_path.suffix.lower() in [".md", ".txt"]:
    print(f"Loading markdown: {input_path.name}")
    paper_text = load_file_text(input_path)
    print(f"   Main paper: {len(paper_text):,} characters")
else:
    raise ValueError(f"Unsupported input type: {input_path}")

# Load SI file if exists
si_text = ""
si_path = find_si_file(input_path)
if si_path:
    print(f"   Found SI file: {si_path.name}")
    try:
        si_text = load_file_text(si_path, pdf_extractor)
        print(f"   SI text: {len(si_text):,} characters")
    except Exception as e:
        print(f"   [WARN] Failed to load SI file: {e}")

# Create Paper object
paper = Paper(
    name=input_path.stem,
    id=input_path.stem,
    publication_text=paper_text,
    si_text=si_text,
)

print(f"\n[OK] Paper loaded: {paper.name}")
print(f"   Main text: {len(paper.publication_text):,} chars")
print(f"   SI text: {len(paper.si_text):,} chars")

In [ ]:
import sys

print(sys.executable)

---
## Step 1: Extract Materials

In [ ]:
from llm_synthesis.transformers.material_extraction.dspy_extraction import (
    DspyTextExtractor,
    make_dspy_text_extractor_signature,
)
from llm_synthesis.utils.dspy_utils import get_llm_from_name
from llm_synthesis.utils.markdown_utils import clean_text

material_sig = make_dspy_text_extractor_signature(
    instructions=(
        "Extract ALL distinct porous material compositions that were synthesized "
        "and tested in this paper. IMPORTANT: If the paper studies multiple variants "
        "(e.g., different doping levels x=0.1, x=0.2, x=0.3), list EACH variant "
        "as a separate material. Focus on materials that were actually synthesized, "
        "not just mentioned or referenced from other works."
    ),
    output_name="materials",
    output_description=(
        "ALL distinct synthesized material compositions as a comma-separated list "
        "using acronyms from the text. "
        "Never merge variants into a single generic name."
    ),
)

material_lm = get_llm_from_name(
    # "gemini-3.0-pro",
    # "gemini-3.0-flash",
    "gemini-2.5-flash-lite",
    model_kwargs={"temperature": 0.0, "max_tokens": 8000},
)
material_extractor = DspyTextExtractor(signature=material_sig, lm=material_lm)

print("Extracting materials...")
materials_text = material_extractor.forward(
    input=clean_text(paper.publication_text)
)

materials = [
    m.strip() for m in materials_text.replace("\n", ",").split(",") if m.strip()
]

print(f"\n{'=' * 60}")
print(f"MATERIALS FOUND ({len(materials)} total)")
print("=" * 60)
for i, mat in enumerate(materials, 1):
    print(f"  {i}. {mat}")

---
## Step 2: Extract Synthesis Procedures

In [ ]:
from llm_synthesis.metrics.judge.general_synthesis_judge import (
    DspyGeneralSynthesisJudge,
    make_general_synthesis_judge_signature,
)
from llm_synthesis.transformers.synthesis_extraction.dspy_synthesis_extraction import (
    DspySynthesisExtractor,
    make_dspy_synthesis_extractor_signature,
)

SYNTHESIS_SYSTEM_PROMPT = """You are a helpful assistant that extracts structured synthesis procedures from scientific papers.

IMPORTANT: For the synthesis_method field, you MUST choose from these exact values:
'PVD', 'CVD', 'arc discharge', 'ball milling', 'spray pyrolysis', 'electrospinning',
'sol-gel', 'hydrothermal', 'solvothermal', 'precipitation', 'coprecipitation', 'combustion',
'microwave-assisted', 'sonochemical', 'template-directed', 'solid-state', 'flux growth',
'float zone & Bridgman', 'arc melting & induction melting', 'spark plasma sintering',
'electrochemical deposition', 'chemical bath deposition', 'liquid-phase epitaxy', 'self-assembly',
'atomic layer deposition', 'molecular beam epitaxy', 'pulsed laser deposition', 'ion implantation',
'lithographic patterning', 'wet impregnation', 'incipient wetness impregnation', 'mechanical mixing',
'solution-based', 'mechanochemical', 'other'

For the target_compound_type field, you MUST choose from these exact values:
'framework & porous materials', 
'hybrid & organic-inorganic', 
'other'

If the exact method is not in the list, use the closest match or 'other'."""

synthesis_sig = make_dspy_synthesis_extractor_signature(
    instructions=(
        "Extract the complete structured synthesis procedure for the specified material. "
        "Include all steps, conditions (temperature, time, atmosphere), equipment, and precursors. "
        "Be thorough and preserve all quantitative details."
    ),
)

synthesis_lm = get_llm_from_name(
    GEMINI_MODEL,
    model_kwargs={"temperature": 0.0, "max_tokens": 32000, "num_retries": 3},
    system_prompt=SYNTHESIS_SYSTEM_PROMPT,
)
synthesis_extractor = DspySynthesisExtractor(
    signature=synthesis_sig, lm=synthesis_lm
)

# Judge
judge_lm = get_llm_from_name(
    GEMINI_MODEL,
    model_kwargs={"temperature": 0.1, "max_tokens": 8000},
)
judge_sig = make_general_synthesis_judge_signature()
judge = DspyGeneralSynthesisJudge(signature=judge_sig, lm=judge_lm)

print("[OK] Synthesis extractor and judge initialized")

In [ ]:
from llm_synthesis.models.paper import SynthesisEntry

all_syntheses = []
text_for_llm = clean_text(paper.publication_text)

for i, material in enumerate(materials, 1):
    print(f"\n{'=' * 60}")
    print(f"EXTRACTING SYNTHESIS {i}/{len(materials)}: {material}")
    print("=" * 60)

    try:
        synthesis = synthesis_extractor.forward(input=(text_for_llm, material))

        try:
            evaluation = judge.forward(
                (text_for_llm, json.dumps(synthesis.model_dump()), material)
            )
            print(f"   [OK] Score: {evaluation.scores.overall_score}/5.0")
        except Exception as e:
            print(f"   [WARN] Judge failed: {e}")
            evaluation = None

        all_syntheses.append(
            SynthesisEntry(
                material=material,
                synthesis=synthesis,
                evaluation=evaluation,
            )
        )
        print(f"   Method: {synthesis.synthesis_method}")
        print(f"   Steps: {len(synthesis.steps)}")

    except Exception as e:
        print(f"   [ERROR] {e}")
        all_syntheses.append(
            SynthesisEntry(
                material=material,
                synthesis=None,
                evaluation=None,
            )
        )

print(f"\n[OK] Extracted synthesis for {len(all_syntheses)} materials")

---
## Step 3: Extract Adsorption Data from Text  ★ NEW

Use an LLM to extract BET surface area and gas adsorption coverage values mentioned in the paper text.
This gives us **text-reported** porosity data per material for comparison with VLM-extracted values from plots.


In [ ]:
# Create a porosity/ BET adsorption text extractor using the same DspyTextExtractor pattern
por_text_sig = make_dspy_text_extractor_signature(
    signature_name="TextToPorosity",
    instructions=(
        "Extract ALL BET surface area values reported in this porous materials paper. "
        "For EACH material that has a BET surface area value mentioned in the text, report:\n"
        "  - The material formula\n"
        "  - coverage, or surface area: the maximum loading per gram of material, or when the plot starts to plateau.\n"
        "  - p_cov (if explicitly reported): pressure where the maximum loading is achieved.\n"
        "  - Whether the material is porous (YES/NO)\n\n"
        "IMPORTANT RULES:\n"
        "1. Only extract values explicitly stated in the text. Do NOT estimate or calculate.\n"
        "2. Most papers report only a SINGLE surface area coverage value without distinguishing onset/mid/zero. "
        "   In that case, report it as 'coverage: <value> m^2/g' and use 'NR' for p_cov.\n"
        "3. Only report p_coverage if the paper EXPLICITLY states it as separate value.\n"
        "4. If a material is mentioned but no coverage is given, report 'porosity: NO' or 'coverage: NR'.\n"
        "5. Keep your reasoning SHORT. Focus on extracting values, not explaining the paper.\n"
        "6. If the pressure is given as p/p0 or with respect to STP (standard temperature and pressure), leave the units as 'p0 atm'."
    ),
    input_description="The full publication text from a porous material paper.",
    output_name="coverage_values",
    output_description=(
        "For each material, one line in the format:\n"
        "material_formula | porous: YES/NO | coverage: <value> m^2/g | p_cov: <value> atm\n"
        "Use 'NR' (not reported) for values not explicitly stated in the text.\n"
        "Examples:\n"
        "  Ba0.6K0.4Fe2As2 | porous: YES | coverage: 36 m^2/g | p_cov: 34 p0 atm\n"
        "  MgB2 | porous: YES | coverage: 1000 m^2/g | p_cov: NR\n"
        "  BaFe2As2 | porous: NO | coverage: 2097 m^2/g | p_cov: NR"
    ),
)

por_text_lm = get_llm_from_name(
    GEMINI_MODEL,
    model_kwargs={"temperature": 0.0, "max_tokens": 16384},
)
por_text_extractor = DspyTextExtractor(signature=por_text_sig, lm=por_text_lm)

# Truncate very long papers to avoid output truncation issues
# DSPy wraps the text in JSON and adds reasoning — keep input reasonable
MAX_TEXT_CHARS = 60_000
if len(text_for_llm) > MAX_TEXT_CHARS:
    print(
        f"[INFO] Paper text is {len(text_for_llm):,} chars — truncating to {MAX_TEXT_CHARS:,} for porosity text extraction"
    )
    por_input_text = text_for_llm[:MAX_TEXT_CHARS]
else:
    por_input_text = text_for_llm

print("Extracting porosity values from text...")
try:
    por_text_raw = por_text_extractor.forward(input=por_input_text)
except Exception as e:
    print(f"[WARN] Porosity text extraction failed: {e}")
    print("[WARN] Falling back to empty Porosity text results")
    por_text_raw = ""

print(f"\n{'=' * 60}")
print("Porosity VALUES FROM TEXT")
print("=" * 60)
print(por_text_raw)

In [ ]:
# Parse the porosity text extraction output into a structured dict
def parse_porosity_text_response(raw_text: str) -> dict:
    """
    Parse the BET/adsorption text extraction response into a dict:
    {material_name: {"porous": bool, "bet_surface_area": float|None, "coverage": float|None}}
    """
    results = {}
    for line in raw_text.strip().split("\n"):
        line = line.strip()
        if not line or "|" not in line:
            continue
        parts = [p.strip() for p in line.split("|")]
        if len(parts) < 2:
            continue

        material = parts[0].strip()
        entry = {
            "porous": False,
            "bet_surface_area": None,
            "coverage": None,
        }

        for part in parts[1:]:
            part_lower = part.lower().strip()
            if "porous" in part_lower:
                entry["porous"] = "yes" in part_lower
            else:
                match = re.match(
                    r"(bet_surface_area|coverage)\s*:\s*(\d+\.?\d*)\s*",
                    part_lower,
                )
                if match:
                    entry[match.group(1)] = float(match.group(2))

        results[material] = entry

    return results


por_from_text = parse_porosity_text_response(por_text_raw)

print(f"Parsed porosity values for {len(por_from_text)} materials:")
for mat, vals in por_from_text.items():
    is_porous = "YES" if vals["porous"] else "NO"
    bet = vals["bet_surface_area"]
    cov = vals["coverage"]
    bet_str = f"{bet:.0f} m²/g" if bet is not None else "NR"
    cov_str = f"{cov:.2f}" if cov is not None else "NR"
    print(f"  {mat}: porous={is_porous}, BET={bet_str}, coverage={cov_str}")

---
## Step 4: Extract Figures

In [ ]:
if SKIP_FIGURES:
    print("[SKIP] Skipping figure extraction")
    figures = []
else:
    from llm_synthesis.transformers.figure_extraction import (
        FigureExtractorMarkdown,
    )

    extractor = FigureExtractorMarkdown(
        segmenter="florence",
        florence_repo_id="amayuelas/plot-visualization-florence-2-lora-32",
    )
    print("Extracting figures using Florence-2...")
    figures = extractor.forward(paper.publication_text)

    print(f"\n{'=' * 60}")
    print(f"FIGURES FOUND ({len(figures)} subfigures)")
    print("=" * 60)
    for i, fig in enumerate(figures):
        print(
            f"  {i + 1}. {fig.figure_reference or f'Figure {i}'}: {fig.figure_class}"
        )

---
## Step 5: Extract Plot Data (Claude VLM)

Send all figures to Claude to extract (pressure, uptake) coordinate data from adsorption isotherm plots.


In [ ]:
if SKIP_FIGURES or not figures:
    print("[SKIP] Skipping plot data extraction")
    plots = []
    plot_figures = []
else:
    from llm_synthesis.models.figure import FigureInfoWithPaper
    from llm_synthesis.transformers.plot_extraction.claude_extraction.plot_data_extraction import (
        ClaudeLinePlotDataExtractor,
    )
    from llm_synthesis.utils.figure_utils import clean_text_from_images

    print(f"Extracting data from {len(figures)} figures using Claude VLM...")

    # Use higher max_tokens so the axis metadata at the end doesn't get truncated
    plot_extractor = ClaudeLinePlotDataExtractor(
        model_name=CLAUDE_MODEL, max_tokens=4096
    )

    plots = []
    plot_figures = []

    for i, fig in enumerate(figures):
        print(
            f"\n  [{i + 1}/{len(figures)}] {fig.figure_reference or f'Figure {i}'} ({fig.figure_class})"
        )

        fig_with_paper = FigureInfoWithPaper(
            base64_data=fig.base64_data,
            alt_text=fig.alt_text,
            position=fig.position,
            context_before=fig.context_before,
            context_after=fig.context_after,
            figure_reference=fig.figure_reference,
            figure_class=fig.figure_class,
            quantitative=fig.quantitative,
            paper_text=clean_text_from_images(paper.publication_text),
            si_text=paper.si_text,
        )

        try:
            plot_data = plot_extractor.forward(fig_with_paper)
            if plot_data and plot_data.name_to_coordinates:
                plots.append(plot_data)
                plot_figures.append(fig)
                series_names = list(plot_data.name_to_coordinates.keys())
                print(f"    [OK] {len(series_names)} series: {series_names}")
                print(
                    f"    Axes: x={plot_data.x_axis_label!r} [{plot_data.x_axis_unit!r}]"
                    f"  y={plot_data.y_left_axis_label!r} [{plot_data.y_left_axis_unit!r}]"
                )
            else:
                print("    [--] No extractable data")
        except Exception as e:
            print(f"    [ERROR] {e}")

    print(f"\n[OK] Extracted data from {len(plots)} plots")
    print(f"   Claude VLM cost: ${plot_extractor.get_cost():.4f}")

---
## Step 6: Filter for Adsorption/Isotherm Plots

Keep only plots that look like adsorption isotherms (pressure vs. gas uptake curves).
Note: `PlotFilterConfig.for_superconductivity()` is used here as a placeholder — this should be replaced
with a porosity-specific filter config (`PlotFilterConfig.for_porosity()`) once implemented.


In [ ]:
from llm_synthesis.config.plot_filter_config import PlotFilterConfig
from llm_synthesis.transformers.performance_linking.plot_filter import (
    PlotFilter,
)

# TODO: replace with PlotFilterConfig.for_porosity() once implemented.
# For now, using the superconductivity config as a structural placeholder.
filter_config = PlotFilterConfig.for_superconductivity()
plot_filter = PlotFilter(filter_config)

print("Plot Filter (placeholder — needs porosity-specific config):")
print(f"   X-axis labels: {filter_config.x_axis_labels}")
print(f"   X-axis units: {filter_config.x_axis_units}")
print(
    f"   Y-axis keywords: {filter_config.y_axis_keywords[:5]}... ({len(filter_config.y_axis_keywords)} total)"
)


def fallback_check_isotherm_plot(plot, fig) -> bool:
    """Fallback: check if a plot with missing axis metadata is an adsorption isotherm
    by looking at the figure caption/context."""
    context = f"{fig.context_before or ''} {fig.context_after or ''} {fig.alt_text or ''}".lower()
    isotherm_hints = [
        "adsorption",
        "isotherm",
        "bet",
        "surface area",
        "uptake",
        "pore",
        "n2 adsorption",
        "co2 adsorption",
        "langmuir",
    ]
    return any(hint in context for hint in isotherm_hints)


def _is_axis_missing(label, unit) -> bool:
    return not label and not unit


if SKIP_FIGURES or not plots:
    print("[SKIP] Skipping plot filtering")
    relevant_plots = []
else:
    relevant_plots = []
    for idx, (plot, fig) in enumerate(zip(plots, plot_figures)):
        keep = plot_filter.forward(plot, fig)
        if not keep:
            keep = fallback_check_isotherm_plot(plot, fig)
        if keep:
            relevant_plots.append((idx, plot))

    print(f"\nKept {len(relevant_plots)} / {len(plots)} plots after filtering.")

---
## Step 7: Extract Adsorption Values from Isotherm Plots via VLM  ★ NEW

For each relevant adsorption isotherm plot, ask Claude to extract key values
(e.g. BET surface area, saturation uptake) directly from the plot.
Note: The prompt template below is inherited from the superconductors pipeline and
should be adapted to ask for adsorption-specific quantities.


In [ ]:
from llm_synthesis.services.llm_api.claude import ClaudeAPIClient

# TODO: this prompt needs to be written for adsorption/BET extraction from isotherm plots.
# The template below is a placeholder — adapt it to ask Claude for:
#   - BET surface area (m²/g) read from plot annotations or axis intercepts
#   - Saturation uptake (mmol/g or cm³/g) at high pressure
#   - Isotherm type (Type I, II, IV, etc.) if determinable
ADSORPTION_VLM_PROMPT_TEMPLATE = """
You are analyzing an adsorption isotherm plot from a porous materials paper.
Your task is to extract quantitative adsorption data for each series shown.

For EACH series/curve in the plot, report:
  - series_name: the label or legend entry for this curve
  - gas: the adsorbate gas (N2, CO2, H2, CH4, etc.) if identifiable
  - temperature_K: measurement temperature in Kelvin if shown
  - max_uptake: maximum uptake value read from the plot (include units)
  - bet_surface_area: BET surface area in m²/g if annotated on the plot
  - isotherm_type: IUPAC isotherm type (I, II, III, IV, V, VI) if determinable

Respond as a JSON list. Example:
[
  {{"series_name": "MOF-5", "gas": "N2", "temperature_K": 77,
    "max_uptake": "800 cm³/g", "bet_surface_area": 3534, "isotherm_type": "I"}}
]

{series_name_instruction}
"""

claude_client = ClaudeAPIClient(model=CLAUDE_MODEL)

adsorption_from_vlm = {}  # {plot_idx: {series_name: {...}}}

if SKIP_FIGURES or not relevant_plots:
    print("[SKIP] Skipping VLM adsorption extraction")
else:
    for plot_idx, plot in relevant_plots:
        fig = plot_figures[plot_idx]
        series_names = list(plot.name_to_coordinates.keys())
        series_name_instruction = (
            f"The series in this plot are named: {series_names}. "
            "Use these exact names in your response."
            if series_names
            else ""
        )
        prompt = ADSORPTION_VLM_PROMPT_TEMPLATE.format(
            series_name_instruction=series_name_instruction
        )
        try:
            raw = claude_client.extract_from_figure(fig, prompt)
            # Parse JSON response
            parsed = json.loads(raw)
            adsorption_from_vlm[plot_idx] = {
                entry["series_name"]: entry for entry in parsed
            }
            print(f"  Plot {plot_idx}: extracted {len(parsed)} series")
        except Exception as e:
            print(f"  Plot {plot_idx}: extraction failed — {e}")
            adsorption_from_vlm[plot_idx] = {}

In [ ]:
# NOTE: the sanity_check_delta_tc function from the superconductors pipeline has been removed.
# If you need post-processing of VLM adsorption results (e.g. flagging outliers,
# cross-validating with text-extracted BET values), add it here.

# Example: flag suspiciously high BET values (> 10,000 m²/g is physically implausible)
def sanity_check_adsorption(vlm_results: dict) -> dict:
    """Flag VLM adsorption results with implausible BET surface area values."""
    checked = {}
    for series_name, vals in vlm_results.items():
        entry = dict(vals)
        bet = vals.get("bet_surface_area")
        if bet is not None and bet > 10000:
            entry["_flag"] = (
                f"BET={bet} m²/g exceeds physical maximum (~7000 m²/g for COFs)"
            )
        checked[series_name] = entry
    return checked


# Apply sanity check to all extracted plots
for plot_idx in list(adsorption_from_vlm.keys()):
    adsorption_from_vlm[plot_idx] = sanity_check_adsorption(
        adsorption_from_vlm[plot_idx]
    )

---
## Step 8: Link Plot Series to Materials

In [ ]:
if SKIP_FIGURES or not relevant_plots:
    print("[SKIP] Skipping performance linking")
    plot_mappings = []
else:
    from llm_synthesis.models.performance import PlotMaterialMapping
    from llm_synthesis.transformers.performance_linking.base import LinkingInput
    from llm_synthesis.transformers.performance_linking.series_material_linker import (
        SeriesMaterialLinker,
    )

    print(f"Linking plot series to {len(materials)} materials...")

    linker_lm = get_llm_from_name(
        LINKER_MODEL,
        model_kwargs={"temperature": 0.0, "max_tokens": 8000},
    )
    series_linker = SeriesMaterialLinker(lm=linker_lm)

    plot_mappings = []

    for idx, plot in relevant_plots:
        fig = plot_figures[idx]
        series_names = list(plot.name_to_coordinates.keys())

        print(f"\n  Plot {idx}: {len(series_names)} series")
        print(f"    Series: {series_names}")

        context = f"{fig.context_before} {fig.context_after}"
        plot_meta = {
            "title": plot.title,
            "x_axis_label": plot.x_axis_label,
            "x_axis_unit": plot.x_axis_unit,
            "y_left_axis_label": plot.y_left_axis_label,
            "y_left_axis_unit": plot.y_left_axis_unit,
        }

        linking_input = LinkingInput(
            materials=materials,
            series_names=series_names,
            context=context,
            plot_metadata=plot_meta,
        )
        validated_mappings = series_linker.forward(linking_input)

        matched_series = {m.series_name for m in validated_mappings}
        unmatched = [s for s in series_names if s not in matched_series]

        plot_mappings.append(
            PlotMaterialMapping(
                plot_index=idx,
                figure_reference=fig.figure_reference,
                mappings=validated_mappings,
                unmatched_series=unmatched,
            )
        )

        for m in validated_mappings:
            print(
                f"    '{m.series_name}' -> '{m.material_name}' ({m.confidence})"
            )
        if unmatched:
            print(f"    [WARN] Unmatched: {unmatched}")

    print("\n[OK] Linking complete")

---
## Step 9: Aggregate & Build Final Results

Combine everything: synthesis + BET/adsorption from text + adsorption from VLM plots + per-material.


In [ ]:
from llm_synthesis.utils.performance_utils import (
    aggregate_all_materials_performance,
)


def _normalize_formula(s: str) -> str:
    base = re.sub(r"\s*\([^)]*\)\s*$", "", s).strip()
    base = base.replace("δ", "delta").replace("Δ", "delta")
    for uni, asc in zip("₀₁₂₃₄₅₆₇₈₉₋", "0123456789-"):
        base = base.replace(uni, asc)
    return base.lower().replace(" ", "").replace("−", "-")


def _find_matching_text_porosity(material: str, por_dict: dict) -> list:
    """Find all porosity text entries whose formula matches `material`."""
    mat_norm = _normalize_formula(material)
    matches = []
    for key, entry in por_dict.items():
        if _normalize_formula(key) == mat_norm:
            matches.append((key, entry))
    return matches


def _find_vlm_adsorption_for_series(
    series_name: str, vlm_results: dict
) -> dict:
    """Fuzzy-match a series name against VLM adsorption result keys."""
    if series_name in vlm_results:
        return vlm_results[series_name]

    def _norm(s):
        for uni, asc in zip("₀₁₂₃₄₅₆₇₈₉₋", "0123456789-"):
            s = s.replace(uni, asc)
        return s.lower().replace(" ", "")

    sn = _norm(series_name)
    for key, val in vlm_results.items():
        vk = _norm(key)
        if sn in vk or vk in sn:
            return val
    if len(vlm_results) == 1:
        only_key, only_val = next(iter(vlm_results.items()))
        print(
            f"    [FALLBACK] Single-series match: '{series_name}' → '{only_key}'"
        )
        return only_val
    return {}


# Aggregate isotherm plot data per material
if plot_mappings and plots:
    performance_data = aggregate_all_materials_performance(
        materials, plot_mappings, plots
    )
else:
    performance_data = {}

# Build VLM adsorption lookup: material → VLM adsorption entry
vlm_adsorption_per_material = {}
for mapping in plot_mappings:
    plot_idx = mapping.plot_index
    if plot_idx not in adsorption_from_vlm:
        continue
    vlm_results_for_plot = adsorption_from_vlm[plot_idx]
    for sm in mapping.mappings:
        vlm_data = _find_vlm_adsorption_for_series(
            sm.series_name, vlm_results_for_plot
        )
        if vlm_data and sm.material_name not in vlm_adsorption_per_material:
            vlm_adsorption_per_material[sm.material_name] = vlm_data

# Fuzzy-match text porosity to materials list
text_porosity_per_material = {}
for material in materials:
    matches = _find_matching_text_porosity(material, por_from_text)
    if matches:
        best_key, best_entry = matches[0]
        text_porosity_per_material[material] = best_entry

# Summary table
print("=" * 80)
print(f"{'Material':<40} {'Porous?':<8} {'BET (text)':>12} {'BET (VLM)':>12}")
print("-" * 80)
for material in materials:
    text_entry = text_porosity_per_material.get(material, {})
    vlm_entry = vlm_adsorption_per_material.get(material, {})
    is_porous = (
        "YES" if text_entry.get("porous") else "NO" if text_entry else "?"
    )
    text_bet = text_entry.get("bet_surface_area")
    vlm_bet = vlm_entry.get("bet_surface_area")
    text_str = f"{text_bet:.0f} m²/g" if text_bet is not None else "NR"
    vlm_str = f"{vlm_bet:.0f} m²/g" if vlm_bet is not None else "NR"
    print(f"{material:<40} {is_porous:<8} {text_str:>12} {vlm_str:>12}")
print("=" * 80)
print(f"\nMaterials with isotherm plot data: {len(performance_data)}")
print(
    f"Materials with VLM BET: {sum(1 for v in vlm_adsorption_per_material.values() if v.get('bet_surface_area') is not None)}"
)
print(
    f"Materials with text BET: {sum(1 for v in text_porosity_per_material.values() if v.get('bet_surface_area') is not None)}"
)

---
## Step 10: Save Results

In [ ]:
from llm_synthesis.utils.performance_utils import sanitize_filename

paper_dir = os.path.join(OUTPUT_DIR, paper.id)
os.makedirs(paper_dir, exist_ok=True)

final_results = []

for entry in all_syntheses:
    mat = entry.material

    text_por_entry = text_porosity_per_material.get(mat, {})
    vlm_por_entry = vlm_adsorption_per_material.get(mat, {})

    result = {
        "material": mat,
        "synthesis": entry.synthesis.model_dump() if entry.synthesis else None,
        "evaluation": entry.evaluation.model_dump()
        if entry.evaluation
        else None,
        "porosity_from_text": text_por_entry if text_por_entry else None,
        "adsorption_from_vlm": vlm_por_entry if vlm_por_entry else None,
        "performance": (
            performance_data[mat].model_dump()
            if mat in performance_data
            else None
        ),
    }
    final_results.append(result)

    mat_name = sanitize_filename(mat)
    mat_path = os.path.join(paper_dir, f"{mat_name}.json")
    with open(mat_path, "w") as f:
        json.dump(result, f, indent=2, default=str)

if plot_mappings:
    with open(os.path.join(paper_dir, "performance_mappings.json"), "w") as f:
        json.dump([m.model_dump() for m in plot_mappings], f, indent=2)

porosity_summary = {
    "paper_id": paper.id,
    "materials": materials,
    "porosity_from_text_raw": por_from_text,
    "porosity_from_text_matched": text_porosity_per_material,
    "adsorption_from_vlm_per_material": vlm_adsorption_per_material,
    "adsorption_from_vlm_raw": {
        str(k): v for k, v in adsorption_from_vlm.items()
    },
}
with open(os.path.join(paper_dir, "porosity_summary.json"), "w") as f:
    json.dump(porosity_summary, f, indent=2, default=str)

summary = {
    "paper_id": paper.id,
    "paper_name": paper.name,
    "total_materials": len(materials),
    "materials_list": materials,
    "total_plots_extracted": len(plots) if not SKIP_FIGURES else 0,
    "isotherm_plots_found": len(relevant_plots) if not SKIP_FIGURES else 0,
    "materials_with_text_bet": sum(
        1
        for v in text_porosity_per_material.values()
        if v.get("bet_surface_area") is not None
    ),
    "materials_with_vlm_adsorption": len(vlm_adsorption_per_material),
    "materials_with_isotherm_data": len(performance_data),
}
with open(os.path.join(paper_dir, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(f"[OK] Results saved to: {paper_dir}/")
print(f"   - {len(final_results)} material files")
print("   - porosity_summary.json")
print("   - summary.json")
if plot_mappings:
    print("   - performance_mappings.json")

---
## Step 11: Export Flat Records + Append to Master CSV

Build one standardized record per material (flat, CSV-friendly) for multi-paper
aggregation. Each run appends to `MASTER_CSV` — duplicates are handled by the
`(paper_id, material)` composite key.

In [ ]:
import csv
from pathlib import Path


def extract_year_from_arxiv_id(paper_id: str) -> int | None:
    clean = paper_id.split("_")[0]
    clean = re.sub(r"v\d+$", "", clean)
    match = re.match(r"^(\d{2})(\d{2})\.\d+$", clean)
    if match:
        yy = int(match.group(1))
        return 2000 + yy if yy < 90 else 1900 + yy
    return None


def normalize_formula_for_csv(s: str) -> str:
    base = re.sub(r"\s*\([^)]*\)\s*$", "", s).strip()
    sub_map = {
        "₀": "0",
        "₁": "1",
        "₂": "2",
        "₃": "3",
        "₄": "4",
        "₅": "5",
        "₆": "6",
        "₇": "7",
        "₈": "8",
        "₉": "9",
        "₋": "-",
        "₊": "+",
    }
    for uni, asc in sub_map.items():
        base = base.replace(uni, asc)
    base = base.replace("δ", "delta").replace("Δ", "Delta")
    return base.replace("−", "-").replace("–", "-").strip()


COLUMNS = [
    "paper_id",
    "year",
    "material",
    "material_normalized",
    "is_porous",
    "bet_surface_area_text",
    "coverage_text",
    "bet_surface_area_vlm",
    "max_uptake_vlm",
    "gas_vlm",
    "temperature_K_vlm",
    "isotherm_type_vlm",
    "has_text_bet",
    "has_vlm_adsorption",
    "synthesis_method",
    "synthesis_score",
]

year = extract_year_from_arxiv_id(paper.id)
flat_records = []

for entry in all_syntheses:
    mat = entry.material

    text_entry = text_porosity_per_material.get(mat, {})
    vlm_entry = vlm_adsorption_per_material.get(mat, {})

    synth_method = entry.synthesis.synthesis_method if entry.synthesis else None
    synth_score = (
        entry.evaluation.scores.overall_score
        if entry.evaluation and entry.evaluation.scores
        else None
    )

    record = {
        "paper_id": paper.id,
        "year": year,
        "material": mat,
        "material_normalized": normalize_formula_for_csv(mat),
        "is_porous": text_entry.get("porous"),
        "bet_surface_area_text": text_entry.get("bet_surface_area"),
        "coverage_text": text_entry.get("coverage"),
        "bet_surface_area_vlm": vlm_entry.get("bet_surface_area"),
        "max_uptake_vlm": vlm_entry.get("max_uptake"),
        "gas_vlm": vlm_entry.get("gas"),
        "temperature_K_vlm": vlm_entry.get("temperature_K"),
        "isotherm_type_vlm": vlm_entry.get("isotherm_type"),
        "has_text_bet": text_entry.get("bet_surface_area") is not None,
        "has_vlm_adsorption": bool(vlm_entry),
        "synthesis_method": synth_method,
        "synthesis_score": synth_score,
    }
    flat_records.append(record)

# Save per-paper JSONL
jsonl_path = os.path.join(paper_dir, "porosity_flat_records.jsonl")
with open(jsonl_path, "w") as f:
    for rec in flat_records:
        f.write(json.dumps(rec, default=str) + "\n")
print(f"[OK] Saved {len(flat_records)} flat records → {jsonl_path}")

# Append to master CSV
master_path = Path(MASTER_CSV)
master_path.parent.mkdir(parents=True, exist_ok=True)

existing_keys = set()
if master_path.exists():
    with open(master_path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            existing_keys.add(
                (row.get("paper_id", ""), row.get("material", ""))
            )

new_records = [
    r
    for r in flat_records
    if (r["paper_id"], r["material"]) not in existing_keys
]
replaced_records = [
    r for r in flat_records if (r["paper_id"], r["material"]) in existing_keys
]

if replaced_records:
    replace_keys = {(r["paper_id"], r["material"]) for r in replaced_records}
    all_rows = []
    if master_path.exists():
        with open(master_path, newline="") as f:
            reader = csv.DictReader(f)
            all_rows = [
                row
                for row in reader
                if (row.get("paper_id", ""), row.get("material", ""))
                not in replace_keys
            ]
    all_rows.extend(
        {k: (str(v) if v is not None else "") for k, v in r.items()}
        for r in flat_records
    )
    with open(master_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=COLUMNS)
        writer.writeheader()
        writer.writerows(all_rows)
    print(
        f"[OK] Master CSV updated (replaced {len(replaced_records)} + added {len(new_records)}) → {master_path}"
    )
else:
    write_header = not master_path.exists() or master_path.stat().st_size == 0
    with open(master_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=COLUMNS)
        if write_header:
            writer.writeheader()
        for rec in flat_records:
            writer.writerow(
                {k: (str(v) if v is not None else "") for k, v in rec.items()}
            )
    print(
        f"[OK] Appended {len(new_records)} records to master CSV → {master_path}"
    )

# Display summary
print(f"\n{'=' * 90}")
print(f"FLAT RECORDS (paper_id={paper.id}, year={year})")
print("=" * 90)
print(
    f"{'Material':<35} {'Porous?':<8} {'BET text':>10} {'BET VLM':>10} {'Gas':>6} {'Synth':<15}"
)
print("-" * 90)
for rec in flat_records:
    por = (
        "YES"
        if rec["is_porous"]
        else ("NO" if rec["is_porous"] is False else "?")
    )
    bet_t = (
        f"{float(rec['bet_surface_area_text']):.0f}"
        if rec["bet_surface_area_text"]
        else "—"
    )
    bet_v = (
        f"{rec['bet_surface_area_vlm']:.0f}"
        if rec["bet_surface_area_vlm"]
        else "—"
    )
    gas = rec["gas_vlm"] or "—"
    synth = rec["synthesis_method"] or "—"
    print(
        f"{rec['material']:<35} {por:<8} {bet_t:>10} {bet_v:>10} {gas:>6} {synth:<15}"
    )

---

## Done!

Results include for each porous material:
- **Synthesis procedure** (GeneralSynthesisOntology)
- **BET surface area from text** (values explicitly reported in the paper, m²/g)
- **Gas adsorption coverage from text** (max loading per gram)
- **Isotherm plot data** (pressure/uptake coordinates linked to materials via VLM)
- **Per-material JSON** + rows appended to master CSV for multi-paper aggregation
